<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/09.Regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 9 — Regularization



Model bahasa bisa "lulus" di data training tanpa belajar sesuatu yang berguna: cukup dengan menghafal. Regularisasi adalah kumpulan cara untuk mencegah itu — menghukum bobot yang besar, mematikan neuron secara acak, menambahkan derau, atau mencari titik minimum yang tidak rapuh. Bab ini membahas semuanya, dan notebook ini menguji setiap teknik dengan cara yang sama.

Resepnya: kita sengaja membuat model kecil menghafal 48 potong teks, lalu mengukur loss-nya di 512 potong teks lain yang tidak pernah ia lihat. Selisih keduanya, *gap*, menunjukkan seberapa parah hafalannya. Setiap teknik kemudian dinilai dari apa yang ia lakukan pada gap itu, pada loss validasi, dan pada ukuran bobot.



## Persiapan

Sel pertama memuat library. Sel kedua adalah gerbang GPU. Ia tidak hanya bertanya "apakah CUDA ada?", tetapi benar-benar mengalokasikan matriks 1024 × 1024 di GPU, mengalikannya, menunggu GPU selesai (`synchronize`), dan memeriksa bahwa hasilnya tidak berisi NaN atau tak hingga. Kalau gagal, notebook berhenti dengan petunjuk; tidak ada perpindahan diam-diam ke CPU. CPU hanya dipakai jika Anda sengaja mengubah `ALLOW_CPU` menjadi `True`, dan saat itu notebook memakai profil kecil dengan step lebih sedikit.

Semua model di bab ini kecil (model utama hanya sekitar 0,37 juta parameter), jadi training dilakukan dalam fp32 di GPU mana pun. Pilihan ini juga menghindari BF16, yang tidak didukung secara native oleh T4.

In [1]:
import gc
import importlib.util
import math
import os
import subprocess
import sys
import warnings

if importlib.util.find_spec("datasets") is None:          # Colab biasanya sudah menyediakan datasets
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])

import numpy as np
import torch
import torch.nn as nn
import transformers
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", message="enable_nested_tensor is True")
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)

python 3.13.16 | torch 2.11.0+cu130 | transformers 5.20.2


In [2]:
# Opt-in CPU harus eksplisit: ganti menjadi True (atau set env CH09_ALLOW_CPU=1) hanya jika memang ingin CPU.
ALLOW_CPU = os.environ.get("CH09_ALLOW_CPU", "0") == "1"


def functional_cuda_device():
    """Kembalikan device CUDA yang terbukti bisa menghitung, atau None jika CUDA tidak ada."""
    if not torch.cuda.is_available():
        return None
    device = torch.device("cuda")
    a = torch.randn(1024, 1024, device=device)
    b = a @ a.T
    torch.cuda.synchronize()
    if not torch.isfinite(b).all().item():
        raise RuntimeError("Uji perkalian matriks di GPU menghasilkan nilai non-finite; GPU tidak dipakai.")
    props = torch.cuda.get_device_properties(device)
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {props.name} | compute capability {props.major}.{props.minor} | "
          f"memori bebas {free / 2**30:.1f} dari {total / 2**30:.1f} GiB | "
          f"BF16 native: {'ya' if props.major >= 8 else 'tidak'}")
    del a, b
    torch.cuda.empty_cache()
    return device


DEVICE = functional_cuda_device()
if DEVICE is None and not ALLOW_CPU:
    raise RuntimeError(
        "GPU CUDA tidak tersedia. Di Colab: Runtime > Change runtime type > pilih GPU (mis. T4), "
        "lalu Runtime > Run all. Untuk sengaja memakai CPU dengan profil kecil, set ALLOW_CPU = True.")
if DEVICE is None:
    DEVICE = torch.device("cpu")
    print("CPU dipakai atas opt-in eksplisit (ALLOW_CPU); profil kecil aktif.")

DTYPE = torch.float32       # model bab ini kecil: fp32 di semua GPU, termasuk T4 yang tidak punya BF16 native
print("device:", DEVICE, "| dtype:", DTYPE)

GPU: Tesla T4 | compute capability 7.5 | memori bebas 14.4 dari 14.6 GiB | BF16 native: tidak
device: cuda | dtype: torch.float32


In [3]:
PROFILES = {
    # default GPU: knob eksperimen asli bab ini, ditambah 3 seed untuk tabel perbandingan
    "gpu": dict(overfit_steps=500, matrix_seeds=(42, 43, 44), encoder_steps=400,
                swa_steps=300, swa_start=200, dp_steps=40),
    # opt-in CPU: semua sel tetap jalan, tetapi rezim hafalannya lebih lemah
    "cpu-kecil": dict(overfit_steps=250, matrix_seeds=(42, 43), encoder_steps=400,
                      swa_steps=150, swa_start=100, dp_steps=8),
}
PROFILE_NAME = "gpu" if DEVICE.type == "cuda" else "cpu-kecil"
P = PROFILES[PROFILE_NAME]
print("profil:", PROFILE_NAME, P)

profil: gpu {'overfit_steps': 500, 'matrix_seeds': (42, 43, 44), 'encoder_steps': 400, 'swa_steps': 300, 'swa_start': 200, 'dp_steps': 40}


## Data: teks kecil yang mudah dihafal

Bagian ini berjalan di CPU karena isinya unduhan dan tokenisasi, bukan komputasi model. Kita mengambil 20.000 baris pertama WikiText-2, memecahnya dengan tokenizer GPT-2, lalu memampatkan kosakata menjadi 4.096 token tersering; semua token lain dilebur menjadi id 0. Kosakata kecil membuat model mini cepat dilatih tanpa mengubah mekanika regularisasi. Teks kemudian dipotong menjadi blok 64 token, 512 blok disisihkan untuk validasi, dan semua blok dipindah ke GPU. Revisi dataset dan tokenizer dipin, jadi angka korpusnya sama di setiap run.

Angka terakhir yang dicetak adalah **patokan unigram**: loss yang dicapai kalau model hanya tahu frekuensi tiap token tanpa melihat konteks. Untuk korpus ini nilainya sekitar 5,71 nats. Satuannya log natural; e^5,71 ≈ 303, artinya tebakan berbasis frekuensi saja kira-kira sebingung memilih acak di antara 303 token. Model yang memakai konteks dengan benar seharusnya turun di bawah angka ini pada data validasi.

In [4]:
SEQ_LEN = 64
VOCAB = 4096

# --- CPU/I/O: unduh + tokenisasi ---
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train",
                       revision="b08601e04326c79dfdd32d625aee71d232d685c3")
tokenizer = AutoTokenizer.from_pretrained("gpt2", revision="607a30d783dfa663caf39e06633721c8d4cfcd7e")
texts = [t for t in dataset[:20000]["text"] if t.strip()]
raw_ids = []
for line_ids in tokenizer(texts)["input_ids"]:
    raw_ids.extend(line_ids + [tokenizer.eos_token_id])
raw_ids = torch.tensor(raw_ids)

# Kosakata 4.096: id 1..4095 untuk token GPT-2 tersering, id 0 untuk semua token lain.
raw_counts = torch.bincount(raw_ids, minlength=len(tokenizer))
top = torch.argsort(raw_counts, descending=True, stable=True)[:VOCAB - 1]
lookup = torch.zeros(len(tokenizer), dtype=torch.long)
lookup[top] = torch.arange(1, VOCAB)
ids = lookup[raw_ids]

n_blocks = len(ids) // SEQ_LEN
blocks = ids[:n_blocks * SEQ_LEN].view(n_blocks, SEQ_LEN)
perm = torch.randperm(n_blocks, generator=torch.Generator().manual_seed(0))

counts = torch.bincount(ids, minlength=VOCAB).double()
probs = counts[counts > 0] / counts.sum()
H_UNIGRAM = float(-(probs * probs.log()).sum())

# --- mulai dari sini semua data training/evaluasi tinggal di DEVICE ---
train_blocks = blocks[perm[:-512]].to(DEVICE)
val_blocks = blocks[perm[-512:]].to(DEVICE)
OVERFIT = train_blocks[:48]                       # 48 blok x 64 token = 3.072 token untuk dihafal
OVERFIT_BATCHES = list(OVERFIT.split(8))          # 6 batch = satu putaran penuh
VAL_BATCHES = list(val_blocks.split(64))          # seluruh 512 blok validasi, 8 batch

print(f"{len(ids):,} token -> {n_blocks:,} blok x {SEQ_LEN} | train {len(train_blocks):,} blok | "
      f"val {len(val_blocks)} blok | data di {train_blocks.device}")
print(f"token yang dilebur ke id 0: {(ids == 0).float().mean():.1%}")
print(f"patokan unigram: {H_UNIGRAM:.3f} nats, setara bingung di antara ~{math.exp(H_UNIGRAM):.0f} token")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

1,302,724 token -> 20,355 blok x 64 | train 19,843 blok | val 512 blok | data di cuda:0
token yang dilebur ke id 0: 16.8%
patokan unigram: 5.714 nats, setara bingung di antara ~303 token


### Satu protokol untuk semua teknik

Supaya perbandingan adil, setiap eksperimen memakai model GPT-2 mini yang sama (2 layer, dimensi 64) dan dimulai dari bobot awal yang identik: model dibuat dari seed tetap di CPU, baru dipindah ke GPU. Model dilatih pada 48 blok yang sama dengan urutan batch yang sama. Satu putaran penuh atas 48 blok butuh 6 step (batch 8), jadi 500 step berarti setiap blok dilihat sekitar 83 kali — resep klasik untuk menghafal.

Empat angka dipakai berulang kali:

- **train loss**: loss pada 48 blok yang dihafal, diukur dalam mode eval (dropout mati);
- **val loss**: loss pada 512 blok validasi yang tidak pernah dilihat;
- **gap** = val − train; makin besar, makin model bergantung pada hafalan;
- **‖w‖**: norma L2 seluruh bobot, untuk melihat apakah sebuah teknik memang mengecilkan bobot.

Selisih loss 1 nat berarti model memberi token yang benar peluang sekitar e ≈ 2,7 kali lebih besar (atau lebih kecil). Sel berikut mendefinisikan protokol itu, lalu mencetak loss model yang belum dilatih sebagai titik nol: nilainya dekat ln 4096 ≈ 8,32, tebakan seragam di antara 4.096 token.

In [5]:
def create_llm(num_layers, hidden_size, num_heads, ff_dim, vocab_size, dropout=0.1, seed=42):
    """GPT-2 kecil dari config (bobot acak, bukan pretrained), langsung di DEVICE.

    Bobot diinisialisasi di CPU dengan seed tetap lalu dipindah, sehingga model dengan seed sama
    selalu mulai dari bobot yang identik. torch.manual_seed juga me-reset RNG CUDA untuk mask dropout."""
    torch.manual_seed(seed)
    config = GPT2Config(n_layer=num_layers, n_embd=hidden_size, n_head=num_heads, n_inner=ff_dim,
                        vocab_size=vocab_size, n_positions=SEQ_LEN,
                        resid_pdrop=dropout, embd_pdrop=dropout, attn_pdrop=dropout)
    return GPT2LMHeadModel(config).to(DEVICE, dtype=DTYPE)


def check_finite(loss, where):
    if not torch.isfinite(loss).item():
        raise FloatingPointError(f"Loss non-finite di {where}; eksperimen dihentikan.")


@torch.no_grad()
def eval_loss(model, batches=None):
    """Rata-rata loss (nats per token) dalam mode eval, jadi dropout mati."""
    model.eval()
    losses = [model(input_ids=b, labels=b).loss for b in (VAL_BATCHES if batches is None else batches)]
    return float(torch.stack(losses).mean())


@torch.no_grad()
def weight_norm(model):
    return float(torch.sqrt(sum((p.float() ** 2).sum() for p in model.parameters())))


def overfit_batch(step):
    return OVERFIT_BATCHES[step % len(OVERFIT_BATCHES)]


def summarize(model, init_norm, steps):
    train = eval_loss(model, OVERFIT_BATCHES)
    val = eval_loss(model)
    if not (math.isfinite(train) and math.isfinite(val)):
        raise FloatingPointError("Loss evaluasi non-finite; eksperimen dihentikan.")
    return {"model": model, "train": train, "val": val, "gap": val - train,
            "wnorm": weight_norm(model), "init_norm": init_norm, "steps": steps}


def train_overfit(steps, lr=1e-3, weight_decay=0.0, dropout=0.0, seed=42, stop_below=None):
    """Protokol rezim hafalan: AdamW pada 48 blok yang sama dengan urutan batch tetap.
    stop_below: berhenti lebih awal begitu train loss <= nilai ini (dicek tiap 10 step)."""
    model = create_llm(2, 64, 2, 256, VOCAB, dropout=dropout, seed=seed)
    init_norm = weight_norm(model)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    for step in range(1, steps + 1):
        model.train()
        x = overfit_batch(step - 1)
        loss = model(input_ids=x, labels=x).loss
        check_finite(loss, "train_overfit")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        if stop_below is not None and step % 10 == 0 and eval_loss(model, OVERFIT_BATCHES) <= stop_below:
            break
    return summarize(model, init_norm, step)


def free_gpu():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


probe = create_llm(2, 64, 2, 256, VOCAB)
N_PARAMS = sum(p.numel() for p in probe.parameters())
print(f"model mini: {N_PARAMS:,} parameter di {next(probe.parameters()).device} | "
      f"loss awal di val: {eval_loss(probe):.3f} (ln {VOCAB} = {math.log(VOCAB):.3f})")
del probe

model mini: 366,336 parameter di cuda:0 | loss awal di val: 8.290 (ln 4096 = 8.318)


## L2 regularization dan weight decay

Ide L2: tambahkan penalti λ/2 · ‖w‖² ke loss. Bobot besar jadi "mahal", sehingga model terdorong memakai bobot kecil, yang biasanya berarti fungsi lebih halus dan kurang menghafal. Buku menyebut L2, ridge regression, dan weight decay sebagai satu hal, lalu memberi loop training yang memakai `AdamW`.

> **Listing buku: loop weight decay**
>
> ```python
> for batch in train_dataloader:
>     optimizer.zero_grad()
>     outputs = model(batch)
>     loss = outputs.loss
> ```
>
> Dengan dataloader berisi dict, `model(batch)` memasukkan seluruh dict sebagai `input_ids`, dan forward GPT-2 gagal dengan `AttributeError: 'dict' object has no attribute 'size'` (teramati). Tanpa `labels`, `outputs.loss` juga akan bernilai `None`. Sel berikut memakai `model(**batch, labels=batch["input_ids"])`.

In [6]:
def train_with_weight_decay(model, train_dataloader, weight_decay=0.01, lr=5e-5, epochs=3):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_dataloader:
            optimizer.zero_grad()
            outputs = model(**batch, labels=batch["input_ids"])     # perbaikan: argumen bernama + labels
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_dataloader):.4f}")


train_dataloader = [{"input_ids": b, "attention_mask": torch.ones_like(b)} for b in OVERFIT_BATCHES]
model = create_llm(num_layers=2, hidden_size=64, num_heads=2, ff_dim=256, vocab_size=VOCAB)
train_with_weight_decay(model, train_dataloader, weight_decay=0.01, lr=3e-3)
del model

Epoch 1, Loss: 7.7345
Epoch 2, Loss: 6.4591
Epoch 3, Loss: 5.6072


Loss rata-rata turun dari epoch ke epoch, jadi loop ini benar-benar melatih model (titik awalnya ln 4096 ≈ 8,32). Learning rate kami naikkan dari 5e-5 di buku menjadi 3e-3: 5e-5 adalah nilai khas untuk fine-tuning model pretrained, sedangkan model ini mulai dari bobot acak dan hanya mendapat 18 step.

Sekarang klaim "L2 = weight decay". Untuk SGD biasa klaim ini benar: gradien penalti λ/2 · ‖w‖² adalah λw, jadi menambah penalti ke loss sama persis dengan mengurangi lr · λ · w dari bobot di setiap step. Adam lain cerita. Adam membagi setiap komponen gradien dengan akar rata-rata kuadratnya sendiri; kalau λw ikut masuk ke gradien, ia ikut dibagi. AdamW (*decoupled weight decay*) memisahkan keduanya: langkah Adam hanya memakai gradien loss, lalu bobot dikalikan (1 − lr · λ) secara terpisah.

Ada juga jebakan nama: argumen `weight_decay` di `torch.optim.Adam` adalah L2 gaya lama (ditambahkan ke gradien), sedangkan di `AdamW` artinya decoupled. Sel berikut menguji tiga pasangan. Supaya yang dibandingkan hanya aturan update optimizer, uji ini sengaja sintetis: satu vektor 10.000 bobot dan lima "gradien loss" yang sudah ditetapkan, sehingga tidak ada efek dari model maupun dari ketidakpastian urutan penjumlahan di GPU. Yang dicetak adalah selisih bobot terbesar setelah 5 step.

In [7]:
# Uji sintetis: bobot dan "gradien loss" ditetapkan dulu, jadi yang dibandingkan hanya aturan update optimizer.
LAM = 0.1
gen = torch.Generator().manual_seed(0)
w_init = (torch.randn(10_000, generator=gen) * 0.02).to(DEVICE)
loss_grads = [(torch.randn(10_000, generator=gen) * 1e-3).to(DEVICE) for _ in range(5)]


def final_weights(make_optimizer, l2_in_loss):
    w = w_init.clone().requires_grad_(True)
    optimizer = make_optimizer([w])
    for g in loss_grads:
        if l2_in_loss:
            w.grad = g + LAM * w.detach()        # gradien dari loss + lambda/2 * ||w||^2
        else:
            w.grad = g.clone()
        optimizer.step()
    return w.detach()


PAIRS = {
    "SGD(weight_decay)   vs SGD + L2 di loss ": (lambda ps: torch.optim.SGD(ps, lr=0.1, weight_decay=LAM),
                                                lambda ps: torch.optim.SGD(ps, lr=0.1)),
    "Adam(weight_decay)  vs Adam + L2 di loss": (lambda ps: torch.optim.Adam(ps, lr=3e-3, weight_decay=LAM),
                                                lambda ps: torch.optim.Adam(ps, lr=3e-3)),
    "AdamW(weight_decay) vs Adam + L2 di loss": (lambda ps: torch.optim.AdamW(ps, lr=3e-3, weight_decay=LAM),
                                                lambda ps: torch.optim.Adam(ps, lr=3e-3)),
}
max_diff = {}
for name, (with_decay, plain) in PAIRS.items():
    max_diff[name] = (final_weights(with_decay, False) - final_weights(plain, True)).abs().max().item()
    print(f"{name}: selisih bobot maksimum setelah 5 step = {max_diff[name]:.1e}")

sgd_diff, adam_diff, adamw_diff = max_diff.values()
assert sgd_diff < 1e-6 and adam_diff < 1e-6 and adamw_diff > 1e-4

SGD(weight_decay)   vs SGD + L2 di loss : selisih bobot maksimum setelah 5 step = 3.7e-09
Adam(weight_decay)  vs Adam + L2 di loss: selisih bobot maksimum setelah 5 step = 7.5e-09
AdamW(weight_decay) vs Adam + L2 di loss: selisih bobot maksimum setelah 5 step = 3.0e-02


Dua pasangan pertama identik: selisihnya nol atau sebatas pembulatan float32. Jadi untuk SGD dan untuk `Adam(weight_decay=...)`, weight decay memang sama dengan L2. Pasangan ketiga berbeda sekitar 0,03, padahal satu langkah Adam berukuran sekitar lr = 0,003; dalam lima step, kedua aturan mendorong sebagian bobot ke arah yang berlawanan. AdamW adalah aturan update yang berbeda, dan itulah yang dipakai listing buku.

Seberapa besar perbedaannya setelah training yang lebih panjang? Tiga model identik dilatih 100 step: tanpa regularisasi, Adam dengan L2 di loss, dan AdamW, keduanya dengan λ = 0,1.

In [8]:
def train_100(mode, steps=100, lr=3e-3):
    model = create_llm(2, 64, 2, 256, VOCAB, dropout=0.0, seed=42)
    decay = LAM if mode == "AdamW weight_decay" else 0.0
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=decay)      # weight_decay=0 sama dengan Adam biasa
    for step in range(steps):
        model.train()
        x = overfit_batch(step)
        loss = model(input_ids=x, labels=x).loss
        if mode == "Adam + L2 di loss":
            loss = loss + 0.5 * LAM * sum((p ** 2).sum() for p in model.parameters())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    return weight_norm(model), eval_loss(model)


start_norm = weight_norm(create_llm(2, 64, 2, 256, VOCAB, dropout=0.0, seed=42))
print(f"norma bobot awal {start_norm:.2f} | peluruhan murni AdamW selama 100 step: "
      f"(1 - lr*lambda)^100 = {(1 - 3e-3 * LAM) ** 100:.3f}")
for mode in ("tanpa regularisasi", "Adam + L2 di loss", "AdamW weight_decay"):
    wn, val = train_100(mode)
    print(f"{mode:<20}: ||w|| {wn:6.2f} | val loss {val:.3f}")
free_gpu()

norma bobot awal 21.29 | peluruhan murni AdamW selama 100 step: (1 - lr*lambda)^100 = 0.970
tanpa regularisasi  : ||w||  42.53 | val loss 6.560
Adam + L2 di loss   : ||w||  13.34 | val loss 6.083
AdamW weight_decay  : ||w||  42.29 | val loss 6.551


Mulai dari baris tanpa regularisasi sebagai pembanding. AdamW dengan λ = 0,1 hanya menggeser norma sedikit: peluruhan murninya adalah faktor (1 − lr · λ) per step, sekitar 0,97 setelah 100 step, seperti yang tercetak di baris pertama. Adam dengan L2 di loss menekan norma jauh lebih kuat. Penyebabnya normalisasi Adam: untuk bobot yang gradien loss-nya kecil — misalnya baris embedding untuk token yang jarang atau tidak pernah muncul di 48 blok — gradiennya hampir seluruhnya λw, dan Adam mengubahnya menjadi langkah berukuran sekitar lr menuju nol, sekecil apa pun λ.

Val loss Adam + L2 juga lebih rendah di run validasi kami. Itu menunjukkan bahwa pada λ yang sama ia bertindak sebagai regularisasi yang jauh lebih kuat, bukan bahwa L2 lebih baik daripada AdamW. Kesimpulan praktisnya: angka λ untuk Adam + L2 dan untuk AdamW tidak bisa dipertukarkan. Pakai `weight_decay` milik AdamW seperti listing buku, dan pilih nilainya khusus untuk AdamW. Apakah weight decay benar-benar menolong generalisasi kita uji nanti, berdampingan dengan dropout, di rezim hafalan yang lebih panjang.

## Dropout

Dropout mematikan sebagian aktivasi secara acak saat training — misalnya 10% di setiap forward pass. Karena tidak ada satu jalur pun yang dijamin hidup, jaringan terpaksa menyebar informasi ke banyak jalur; efeknya mirip melatih banyak sub-jaringan sekaligus. Buku mengimplementasikannya pada transformer kecil buatan sendiri.

> **Listing buku: `TransformerWithDropout` tanpa import**
>
> ```python
> class TransformerWithDropout(nn.Module):
>     def __init__(self, vocab_size, d_model, nhead, num_layers, dropout=0.1):
>         super().__init__()
>         self.embedding = nn.Embedding(vocab_size, d_model)
> ```
>
> Listing di bab ini memakai `torch` dan `nn` tanpa pernah meng-import-nya, sehingga baris pertama sudah gagal dengan `NameError: name 'nn' is not defined` (teramati). Isi kelasnya sendiri benar; sel berikut memakai kelas yang sama, dengan `import torch.nn as nn` di sel persiapan.

Ukuran yang dipakai buku (d_model 768, 12 layer, kosakata GPT-2 50.257 token) cukup besar, jadi kita hanya menghitung parameternya di device `meta`: PyTorch membuat struktur model tanpa mengalokasikan memori untuk bobotnya.

In [9]:
class TransformerWithDropout(nn.Module):                  # kelas dari buku, tidak diubah
    def __init__(self, vocab_size, d_model, nhead, num_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = nn.Embedding(1000, d_model)  # Simplified positional encoding
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=4 * d_model, dropout=dropout),
            num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.embedding(x) + self.pos_encoder(torch.arange(x.size(1), device=x.device))
        x = self.dropout(x)
        x = x.transpose(0, 1)  # Transform to shape expected by transformer
        x = self.transformer(x)
        x = x.transpose(0, 1)  # Transform back
        return self.fc_out(x)


with torch.device("meta"):                                # hanya struktur; tidak ada memori bobot
    full_size = TransformerWithDropout(vocab_size=50257, d_model=768, nhead=12, num_layers=12, dropout=0.1)
n_full = sum(p.numel() for p in full_size.parameters())
print(f"Model parameters: {n_full:,} | fc_out sendiri: {768 * 50257 + 50257:,}")
assert n_full == 163_067_473
del full_size

Model parameters: 163,067,473 | fc_out sendiri: 38,647,633


Hasilnya 163.067.473 parameter, lebih banyak dari GPT-2 small (sekitar 124 juta). Selisih terbesar berasal dari `fc_out`, yang tidak berbagi bobot dengan embedding: satu matriks 768 × 50.257 tambahan, sekitar 38,6 juta parameter. GPT-2 memakai *weight tying*, sehingga matriks itu tidak dihitung dua kali. Dropout terpasang di tiga tempat: setelah embedding, di attention, dan di feed-forward setiap layer.

Masalah yang lebih penting justru tidak memunculkan error. `nn.TransformerEncoder` yang dipanggil tanpa `mask` memakai attention dua arah: setiap posisi bisa melihat token sesudahnya, termasuk token yang sedang ia tebak. Untuk model prediksi-token-berikutnya, itu sama dengan membocorkan jawaban. Kebocoran bisa dideteksi dari angkanya. Sel berikut melatih versi mini kelas ini (dimensi 64, 2 layer) dua kali dengan bobot awal, data, dan urutan batch yang sama — tanpa dan dengan *causal mask*, yaitu mask yang menutup semua posisi di sebelah kanan. Karena `forward` buku tidak menerima mask, forward ditulis ulang di dalam fungsi dengan komponen yang sama.

In [10]:
# === UNDER THE HOOD: loss yang "terlalu bagus" sebagai alarm kebocoran ===
def train_encoder_lm(use_causal_mask, steps, lr=3e-3, seed=42):
    torch.manual_seed(seed)
    enc = TransformerWithDropout(vocab_size=VOCAB, d_model=64, nhead=2, num_layers=2, dropout=0.0).to(DEVICE)
    optimizer = AdamW(enc.parameters(), lr=lr)
    mask = nn.Transformer.generate_square_subsequent_mask(SEQ_LEN - 1).to(DEVICE) if use_causal_mask else None
    data_gen = torch.Generator().manual_seed(seed)          # urutan batch identik untuk kedua varian

    def lm_loss(x):
        inp, tgt = x[:, :-1], x[:, 1:]
        h = enc.dropout(enc.embedding(inp) + enc.pos_encoder(torch.arange(inp.size(1), device=DEVICE)))
        h = enc.transformer(h.transpose(0, 1), mask=mask).transpose(0, 1)
        return nn.functional.cross_entropy(enc.fc_out(h).reshape(-1, VOCAB), tgt.reshape(-1))

    for step in range(steps):
        enc.train()
        idx = torch.randint(0, len(train_blocks), (8,), generator=data_gen).to(DEVICE)
        loss = lm_loss(train_blocks[idx])
        check_finite(loss, "encoder LM")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    enc.eval()
    with torch.no_grad():
        return float(torch.stack([lm_loss(b) for b in VAL_BATCHES]).mean())


va_leaky = train_encoder_lm(use_causal_mask=False, steps=P["encoder_steps"])
va_causal = train_encoder_lm(use_causal_mask=True, steps=P["encoder_steps"])
print(f"patokan unigram (tanpa konteks)      : {H_UNIGRAM:.3f}")
print(f"encoder TANPA causal mask (buku)     : val loss {va_leaky:.3f}")
print(f"encoder DENGAN causal mask           : val loss {va_causal:.3f}")
free_gpu()

patokan unigram (tanpa konteks)      : 5.714
encoder TANPA causal mask (buku)     : val loss 2.684
encoder DENGAN causal mask           : val loss 5.033


Model dengan causal mask hanya boleh memakai konteks di kirinya. Val loss-nya turun di bawah patokan unigram, dan selisih itu adalah manfaat konteks yang sah. Model tanpa mask turun jauh lebih rendah lagi. Itu bukan tanda model lebih pintar: posisi *t* cukup melihat token *t+1* di input lalu menyalinnya. Model seperti itu akan runtuh saat dipakai untuk generate, ketika token berikutnya memang belum ada. Pelajarannya: val loss yang terlalu bagus untuk ukuran model dan lama training adalah alarm kebocoran, bukan prestasi. GPT-2 dari `create_llm` aman karena causal mask sudah tertanam di arsitekturnya.

Satu klaim lain di teks buku juga perlu dicek: saat inference, katanya, "*the weights are typically scaled by the dropout rate*". Itu cara kerja dropout versi awal. PyTorch memakai **inverted dropout**: penskalaan dilakukan saat training — unit yang selamat dikalikan 1/(1 − p) — dan saat eval dropout tidak melakukan apa pun. Kita periksa dengan p = 0,5 pada 100.000 angka 1 di GPU:

In [11]:
drop = nn.Dropout(p=0.5)
x = torch.ones(100_000, device=DEVICE)
drop.train()
y_train = drop(x)
drop.eval()
y_eval = drop(x)

frac_zero = (y_train == 0).float().mean().item()
survivors = y_train[y_train > 0].unique().tolist()
print(f"train: {frac_zero:.1%} nol | nilai unit yang selamat {survivors} | rata-rata {y_train.mean().item():.4f}")
print(f"eval : identik dengan input = {torch.equal(y_eval, x)}")
assert survivors == [2.0] and torch.equal(y_eval, x) and abs(frac_zero - 0.5) < 0.01

train: 50.1% nol | nilai unit yang selamat [2.0] | rata-rata 0.9982
eval : identik dengan input = True


Saat training sekitar separuh nilai menjadi nol dan sisanya bernilai tepat 2,0 = 1/(1 − 0,5), sehingga rata-ratanya tetap sekitar 1, sama seperti tanpa dropout. Saat eval, output identik dengan input. Buku benar soal tujuannya, yaitu menjaga besaran output konsisten antara training dan inference, tetapi letak penskalaannya terbalik. Konsekuensi praktisnya: `model.eval()` wajib dipanggil sebelum evaluasi supaya dropout benar-benar mati. Fungsi `eval_loss` di notebook ini selalu melakukannya.

### Under the hood: weight decay vs dropout di rezim hafalan

Sekarang pertanyaan utamanya: apakah weight decay dan dropout benar-benar menahan hafalan? Empat konfigurasi dilatih dengan protokol yang sama: tanpa regularisasi, weight decay 1,0, dropout 0,3, dan keduanya. Weight decay 1,0 terdengar besar, tetapi di AdamW efek per step-nya hanya lr × λ = 0,001; tanpa gradien yang melawan, 500 step setara faktor penyusutan e^−0,5 ≈ 0,61. Setiap konfigurasi diulang dengan beberapa seed (profil GPU: 3 seed). Dengan seed yang sama, keempat konfigurasi mulai dari bobot yang identik, dan `assert` di akhir sel memeriksanya. Tabel mencetak rata-rata dan rentang [min–max] antar-seed; baris di bawahnya membandingkan tiap teknik dengan model tanpa regularisasi.

In [12]:
CONFIGS = {
    "tanpa regularisasi": dict(weight_decay=0.0, dropout=0.0),
    "weight decay 1.0": dict(weight_decay=1.0, dropout=0.0),
    "dropout 0.3": dict(weight_decay=0.0, dropout=0.3),
    "keduanya": dict(weight_decay=1.0, dropout=0.3),
}
runs = {name: [] for name in CONFIGS}
for seed in P["matrix_seeds"]:
    for name, kwargs in CONFIGS.items():
        runs[name].append(train_overfit(P["overfit_steps"], seed=seed, **kwargs))


def mean_range(values):
    return f"{np.mean(values):6.3f} [{min(values):5.2f}-{max(values):5.2f}]"


print(f"patokan unigram {H_UNIGRAM:.3f} | {len(P['matrix_seeds'])} seed x {P['overfit_steps']} step per konfigurasi")
print(f"{'konfigurasi':<19} {'train':>21} {'val':>21} {'gap':>21} {'||w||':>7}")
for name, rs in runs.items():
    print(f"{name:<19} {mean_range([r['train'] for r in rs]):>21} {mean_range([r['val'] for r in rs]):>21} "
          f"{mean_range([r['gap'] for r in rs]):>21} {np.mean([r['wnorm'] for r in rs]):7.2f}")

print()
base = runs["tanpa regularisasi"]
for name in list(CONFIGS)[1:]:
    d_gap = np.mean([r["gap"] for r in runs[name]]) - np.mean([r["gap"] for r in base])
    d_val = np.mean([r["val"] for r in runs[name]]) - np.mean([r["val"] for r in base])
    wins = sum(r["val"] < b["val"] for r, b in zip(runs[name], base))
    print(f"{name:<18}: gap {d_gap:+.2f} nats, val {d_val:+.2f} nats vs tanpa regularisasi | "
          f"val lebih baik di {wins}/{len(base)} seed")

# keadilan: dengan seed yang sama, keempat konfigurasi mulai dari bobot yang identik
for i in range(len(P["matrix_seeds"])):
    assert len({runs[name][i]["init_norm"] for name in CONFIGS}) == 1

BASELINE = base[0]                     # seed 42, dipakai lagi di bagian SAM dan FGSM
for rs in runs.values():
    for r in rs:
        if r is not BASELINE:
            r.pop("model")
free_gpu()

patokan unigram 5.714 | 3 seed x 500 step per konfigurasi
konfigurasi                         train                   val                   gap   ||w||
tanpa regularisasi    0.404 [ 0.39- 0.41]   7.986 [ 7.84- 8.24]   7.582 [ 7.43- 7.83]   65.63
weight decay 1.0      1.650 [ 1.59- 1.71]   6.652 [ 6.53- 6.73]   5.002 [ 4.82- 5.12]   58.86
dropout 0.3           1.704 [ 1.64- 1.75]   7.333 [ 7.31- 7.35]   5.629 [ 5.58- 5.67]   62.51
keduanya              2.711 [ 2.65- 2.74]   6.417 [ 6.40- 6.44]   3.706 [ 3.66- 3.76]   55.36

weight decay 1.0  : gap -2.58 nats, val -1.33 nats vs tanpa regularisasi | val lebih baik di 3/3 seed
dropout 0.3       : gap -1.95 nats, val -0.65 nats vs tanpa regularisasi | val lebih baik di 3/3 seed
keduanya          : gap -3.88 nats, val -1.57 nats vs tanpa regularisasi | val lebih baik di 3/3 seed


Baca baris tanpa regularisasi dulu. Train loss turun jauh di bawah patokan unigram — model hafal 48 bloknya — sementara val loss naik melewati patokan itu. Artinya hafalan bukan sekadar tidak membantu: di teks baru, model ini menebak lebih buruk daripada tabel frekuensi tanpa konteks, karena ia memaksakan pola dari 48 blok yang dihafalnya.

Lalu baris lainnya. Weight decay dan dropout sama-sama menaikkan train loss (hafalan tertahan) dan, pada validasi lokal kami, menurunkan val loss dan gap; kombinasi keduanya memberi gap terkecil. Sebelum menyimpulkan dari tabel Anda sendiri, bandingkan selisih antar-konfigurasi dengan lebar rentang [min–max]: selisih yang lebih kecil dari variasi antar-seed belum bisa dibedakan dari kebetulan. Kolom "val lebih baik di k/n seed" membantu membaca hal itu.

Batasnya juga perlu diingat. Ini rezim ekstrem yang sengaja dibuat (48 blok diputar sekitar 83 kali); di profil `cpu-kecil` yang lebih pendek, hafalannya lebih lemah dan selisih antar-konfigurasi ikut mengecil. Nilai λ dan p yang bekerja di sini tidak otomatis cocok untuk model besar dan data besar; di sana keduanya perlu di-tune ulang.

## Layer-wise adaptive regularization

Idenya: layer bawah menangkap pola umum dan cukup diberi regularisasi ringan, sedangkan layer atas lebih rawan menghafal detail sehingga diberi regularisasi lebih berat. Buku membungkus model dasar dan menaikkan dropout secara linear per layer: 0,10 di layer pertama, lalu +0,02 di setiap layer berikutnya.

> **Listing buku: wrapper layer-wise**
>
> ```python
> dropout = self.base_dropout
> + i * self.dropout_increase_per_layer
> layer.attn.dropout.p = dropout
> ...
> def forward(self, *args, kwargs):
>     return self.base_model(*args, kwargs)
> base_model = create_lm_model()
> ```
>
> Tiga error teramati: `create_lm_model` tidak didefinisikan (`NameError`); blok attention GPT-2 tidak punya `attn.dropout` — yang ada `attn_dropout` dan `resid_dropout` (`AttributeError`); dan `kwargs` tanpa `**` membuat `model(input_ids=...)` ditolak (`TypeError`). Masalah keempat paling berbahaya karena tidak memunculkan error: baris `+ i * ...` adalah pernyataan yang terpisah.

Contoh tandingan kecil berikut sengaja meniru dua baris pertama itu:

In [13]:
# Contoh tandingan yang disengaja: bentuk baris seperti di buku. Tidak error, tetapi kenaikannya hilang.
def dropout_as_printed(i, base_dropout=0.1, increase=0.02):
    dropout = base_dropout
    + i * increase
    return dropout


print([dropout_as_printed(i) for i in range(4)])

[0.1, 0.1, 0.1, 0.1]


Semua layer mendapat 0,1. Python menerima kedua baris itu: `+ i * increase` adalah ekspresi yang sah, dihitung, lalu dibuang. Tidak ada traceback, dan fitur yang menjadi judul bagian ini diam-diam tidak pernah aktif. Satu-satunya pertahanan adalah mencetak dan memeriksa state yang Anda kira sudah berubah. Versi yang bekerja di bawah memperbaiki keempat masalah, lalu membuktikan jadwalnya dengan `assert`. Model dasarnya GPT-2 mini 12 layer, supaya kedua belas laju bisa terlihat.

In [14]:
class LayerwiseAdaptiveRegularization(nn.Module):
    def __init__(self, base_model, num_layers, base_dropout=0.1, dropout_increase_per_layer=0.02):
        super().__init__()
        self.base_model = base_model
        self.num_layers = num_layers
        self.base_dropout = base_dropout
        self.dropout_increase_per_layer = dropout_increase_per_layer
        self.set_layerwise_dropout()

    def set_layerwise_dropout(self):
        for i, layer in enumerate(self.base_model.transformer.h):
            dropout = (self.base_dropout
                       + i * self.dropout_increase_per_layer)     # tanda kurung: satu ekspresi utuh
            layer.attn.attn_dropout.p = dropout                   # nama atribut GPT-2 yang benar
            layer.attn.resid_dropout.p = dropout
            layer.mlp.dropout.p = dropout

    def forward(self, *args, **kwargs):                            # ** yang hilang dikembalikan
        return self.base_model(*args, **kwargs)


base_model = create_llm(num_layers=12, hidden_size=48, num_heads=2, ff_dim=192, vocab_size=VOCAB)
model = LayerwiseAdaptiveRegularization(base_model, num_layers=12)
rates = [round(layer.mlp.dropout.p, 2) for layer in base_model.transformer.h]
print("dropout per layer:", rates)
assert rates == [round(0.1 + 0.02 * i, 2) for i in range(12)]

model.train()
x = OVERFIT_BATCHES[0]
loss = model(input_ids=x, labels=x).loss
print(f"forward lewat wrapper (mode train, dropout aktif): loss {loss.item():.3f}")
del base_model, model, loss
free_gpu()

dropout per layer: [0.1, 0.12, 0.14, 0.16, 0.18, 0.2, 0.22, 0.24, 0.26, 0.28, 0.3, 0.32]
forward lewat wrapper (mode train, dropout aktif): loss 8.290


Kedua belas laju dari 0,10 sampai 0,32 kini benar-benar terpasang, dan forward dengan argumen bernama mengalir lewat wrapper. Loss-nya masih sekitar 8,3 karena model ini belum dilatih. Bagian ini memeriksa mekanikanya, bukan manfaatnya: apakah jadwal per-layer lebih baik daripada dropout seragam tidak diuji di sini. Untuk mengujinya, latih wrapper ini dengan protokol rezim hafalan dan bandingkan dengan dropout seragam yang rata-ratanya sama.

## Gradient clipping dan noise injection

Gradient clipping membatasi panjang vektor gradien: jika normanya melebihi ambang (di sini 1,0), seluruh vektor diperkecil sampai normanya tepat 1,0. Tujuan utamanya stabilitas; dari sisi regularisasi, satu batch tidak bisa menarik bobot terlalu jauh. Noise injection menambahkan derau kecil ke input, bobot, atau aktivasi, supaya model tidak bergantung pada pola yang terlalu spesifik. Buku menggabungkan keduanya dalam satu loop.

> **Listing buku: noise pada token id**
>
> ```python
> import torch.nn.functional as F
> noise = torch.randn_like(input_ids, dtype=torch.float) * noise_factor
> noisy_inputs = input_ids.float() + noise
> noisy_inputs = noisy_inputs.long().clamp(min=0, max=model.config.vocab_size - 1)
> clip_grad_norm_(model.parameters(), grad_clip)
> ```
>
> `clip_grad_norm_` dipakai tanpa di-import (`NameError`, teramati), sementara `F` yang di-import tidak pernah dipakai. Masalah yang lebih dalam tidak memunculkan error: derau ditambahkan ke *nomor* token. Sel berikut mengukur apa yang sebenarnya terjadi pada 12.800 token (200 blok).

In [15]:
# === UNDER THE HOOD: "noise kecil" pada token id ===
# Hanya aritmetika tensor kecil tanpa training, jadi dijalankan di CPU.
sample_ids = train_blocks[:200].flatten().cpu()
gen = torch.Generator().manual_seed(0)
noise_stats = {}
for noise_factor in (0.01, 0.1, 0.5):
    noise = torch.randn(sample_ids.shape, generator=gen) * noise_factor
    via_long = (sample_ids.float() + noise).long().clamp(0, VOCAB - 1)       # seperti listing buku
    via_round = (sample_ids.float() + noise).round().long().clamp(0, VOCAB - 1)
    changed = via_long != sample_ids
    noise_stats[noise_factor] = {
        "long": changed.float().mean().item(),
        "to_prev": (via_long[changed] == sample_ids[changed] - 1).float().mean().item(),
        "round": (via_round != sample_ids).float().mean().item(),
    }
    s = noise_stats[noise_factor]
    print(f"noise_factor {noise_factor:<4}: .long() mengubah {s['long']:6.1%} token "
          f"({s['to_prev']:.0%} di antaranya menjadi id-1) | .round() mengubah {s['round']:6.2%}")

nonzero = (sample_ids != 0).float().mean().item()
print(f"porsi token ber-id bukan 0: {nonzero:.1%}; separuhnya: {nonzero / 2:.1%}")
assert noise_stats[0.01]["to_prev"] == 1.0 and noise_stats[0.01]["round"] == 0.0

noise_factor 0.01: .long() mengubah  42.0% token (100% di antaranya menjadi id-1) | .round() mengubah  0.00%
noise_factor 0.1 : .long() mengubah  40.9% token (100% di antaranya menjadi id-1) | .round() mengubah  0.00%
noise_factor 0.5 : .long() mengubah  43.5% token (91% di antaranya menjadi id-1) | .round() mengubah 29.34%
porsi token ber-id bukan 0: 82.6%; separuhnya: 41.3%


Niatnya perturbasi kecil; hasilnya korupsi besar. `.long()` memotong ke arah nol, jadi token 571 ditambah derau −0,004 menjadi 570,996 lalu terpotong ke 570 — token lain sama sekali. Setiap derau negatif, sekecil apa pun, menggeser id turun satu. Separuh derau bernilai negatif, jadi sekitar separuh token berubah. Yang kebal hanya token ber-id 0, karena 0 dikurangi pecahan tetap terpotong ke 0. Itu sebabnya persentase pada noise 0,01 hampir sama dengan "separuh dari porsi token ber-id bukan 0" di baris terakhir.

`.round()` gagal ke arah sebaliknya: pada noise 0,01 tidak ada satu token pun yang berubah, jadi tidak ada perturbasi sama sekali. Keduanya berakar pada salah konsep yang sama: id token adalah label kategori, bukan besaran. Token 570 dan 571 tidak "berdekatan" maknanya. Derau yang bermakna untuk teks ditaruh di ruang embedding, tempat jarak memang mencerminkan kemiripan.

Versi yang bekerja: import `clip_grad_norm_`, tambahkan derau Gaussian sebesar 5% simpangan baku embedding lewat `inputs_embeds` (jalur input yang memang disediakan GPT-2), dan catat norma gradien sebelum dipotong.

In [16]:
from torch.nn.utils import clip_grad_norm_


def train_with_grad_clip_and_noise(model, train_dataloader, grad_clip=1.0, noise_factor=0.05, lr=3e-3, epochs=3):
    optimizer = AdamW(model.parameters(), lr=lr)
    norms = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_dataloader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"]
            embd = model.transformer.wte(input_ids)
            noise = torch.randn_like(embd) * noise_factor * embd.std().detach()
            outputs = model(inputs_embeds=embd + noise, labels=input_ids)       # derau di ruang embedding
            loss = outputs.loss
            loss.backward()
            norms.append(clip_grad_norm_(model.parameters(), grad_clip, error_if_nonfinite=True).item())
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_dataloader):.4f}")
    norms = np.array(norms)
    print(f"norma gradien sebelum clipping: median {np.median(norms):.2f}, maks {norms.max():.2f} | "
          f"{(norms > grad_clip).mean():.0%} dari {len(norms)} step dipotong ke {grad_clip}")


model_noise = create_llm(2, 64, 2, 256, VOCAB)
train_with_grad_clip_and_noise(model_noise, train_dataloader)
del model_noise
free_gpu()

Epoch 1, Loss: 7.7366
Epoch 2, Loss: 6.4652
Epoch 3, Loss: 5.6092
norma gradien sebelum clipping: median 1.56, maks 2.58 | 100% dari 18 step dipotong ke 1.0


Loss tetap turun, jadi derau sebesar ini tidak menghalangi belajar. Baris terakhir memberi arti pada clipping: `clip_grad_norm_` mengembalikan norma *sebelum* dipotong, sehingga kita tahu berapa persen step yang benar-benar dibatasi. Jika hampir semua step dipotong — seperti di validasi lokal kami, dengan median norma sekitar 1,6 — ambang 1,0 aktif mengatur ukuran setiap langkah; jika hampir tidak ada yang dipotong, clipping hanya menjadi jaring pengaman. Knob diubah dari buku (noise 0,01 → 0,05, lr 5e-5 → 3e-3) dengan alasan yang sama seperti sebelumnya. Sel ini memeriksa mekanika, bukan membuktikan bahwa noise memperbaiki generalisasi; untuk itu, bandingkan val loss dengan dan tanpa noise memakai protokol rezim hafalan.

## Regularisasi saat fine-tuning

Fine-tuning model pretrained pada data kecil butuh keseimbangan: regularisasi yang cukup untuk mencegah hafalan, tetapi tidak sampai menghambat adaptasi ke task baru. Resep buku: mulai dengan dropout 0,1, lalu turunkan secara linear setiap epoch.

> **Listing buku: fine-tuning dengan dropout menurun**
>
> ```python
> model = GPT2LMHeadModel.from_pretrained(pretrained_model_name)
> ...
> for batch in train_dataloader:
>     optimizer.zero_grad()
>     outputs = model(batch)
> ```
>
> Kesalahannya sama dengan listing weight decay: `model(batch)` gagal dengan `AttributeError: 'dict' object has no attribute 'size'` (teramati pada run lama yang memakai GPT-2 124M asli, dan pada uji ulang lokal dengan model mini). Bagian penjadwalan dropout-nya benar dan dipakai di sel berikut, yang menerima objek model alih-alih nama model.

Default notebook memakai model mini berbobot acak sebagai pengganti "pretrained", karena yang diuji di sini adalah mekanika jadwal dropout. Versi dengan GPT-2 124M asli tersedia sebagai opsi sesudahnya.

In [17]:
def fine_tune_with_adaptive_regularization(model, train_dataloader, initial_dropout=0.1, epochs=3, lr=3e-3):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    n_drop = sum(isinstance(m, nn.Dropout) for m in model.modules())
    print(f"{n_drop} modul nn.Dropout dikendalikan")
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        current_dropout = initial_dropout * (1 - epoch / epochs)
        for module in model.modules():
            if isinstance(module, nn.Dropout):
                module.p = current_dropout
        for batch in train_dataloader:
            optimizer.zero_grad()
            outputs = model(**batch, labels=batch["input_ids"])     # perbaikan yang sama dengan listing weight decay
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_dataloader):.4f}, Dropout: {current_dropout:.4f}")


model_ft = create_llm(2, 64, 2, 256, VOCAB)          # pengganti mini berbobot acak, bukan pretrained
fine_tune_with_adaptive_regularization(model_ft, train_dataloader)
del model_ft
free_gpu()

7 modul nn.Dropout dikendalikan
Epoch 1, Loss: 7.7345, Dropout: 0.1000
Epoch 2, Loss: 6.4496, Dropout: 0.0667
Epoch 3, Loss: 5.5876, Dropout: 0.0333


Laju dropout turun 0,1 → 0,067 → 0,033, dan `model.modules()` menjangkau ketujuh modul dropout GPT-2 mini — satu setelah embedding, lalu attention, residual, dan MLP di masing-masing dari 2 layer — tanpa perlu tahu namanya. Satu koreksi istilah: yang diimplementasikan buku adalah *scheduled* (annealed) dropout, yaitu laju yang turun mengikuti waktu dan sama untuk semua unit. Istilah "adaptive dropout" di literatur (Ba & Frey, 2013) merujuk teknik lain: laju drop per unit yang dipelajari oleh jaringan tambahan.

**Opsional: GPT-2 124M pretrained asli.** Ubah `RUN_PRETRAINED_GPT2` menjadi `True` untuk fine-tuning dengan bobot pretrained (unduhan sekitar 550 MB, GPU disarankan). Kosakata 4.096 kita adalah hasil pemetaan ulang, jadi versi ini memakai id token GPT-2 asli dari 48 blok teks yang sama, dengan lr 5e-5 seperti di buku. Jalur ini belum diuji di Colab; secara lokal ia hanya diuji dengan GPT-2 berbobot acak berkosakata penuh sebagai pengganti.

In [18]:
RUN_PRETRAINED_GPT2 = False      # True: unduh dan fine-tune GPT-2 124M asli

if RUN_PRETRAINED_GPT2:
    raw_blocks = raw_ids[:n_blocks * SEQ_LEN].view(n_blocks, SEQ_LEN)[perm[:48]].to(DEVICE)   # teks = OVERFIT
    raw_loader = [{"input_ids": b, "attention_mask": torch.ones_like(b)} for b in raw_blocks.split(8)]
    gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE, dtype=DTYPE)
    fine_tune_with_adaptive_regularization(gpt2, raw_loader, lr=5e-5)
    del gpt2, raw_blocks, raw_loader
    free_gpu()
else:
    print("Dilewati: RUN_PRETRAINED_GPT2 = False (tidak dijalankan).")

Dilewati: RUN_PRETRAINED_GPT2 = False (tidak dijalankan).


## Teknik regularisasi yang lebih baru

Lima teknik berikut tidak hanya menahan hafalan, tetapi juga memengaruhi *ke mana* optimizer membawa bobot: ke tengah lembah loss (SWA), ke daerah yang landai (SAM), ke gradien yang tidak didominasi satu contoh (DP), menjauhi arah yang rapuh (FGSM), atau ke lintasan yang lebih halus (Lookahead).

### Stochastic weight averaging (SWA)

Pada fase akhir training, SWA memakai learning rate konstan yang cukup besar sehingga bobot "memantul" di sekitar dasar lembah. Alih-alih memakai titik terakhir, SWA merata-ratakan bobot dari banyak titik pantulan; rata-ratanya cenderung jatuh lebih dekat ke tengah lembah. Snippet buku hanya fragmen tanpa model dan loop, jadi kerangkanya kita lengkapi dengan `AveragedModel` dan `SWALR` dari PyTorch. Pada profil GPU, fase SWA dimulai di step 200 dari 300 (buku: epoch 75 dari 100), dan lr naik dari 0,003 ke 0,01 (buku: `swa_lr` 0,05). Sebagai pembanding, val loss juga dicatat tepat saat fase SWA dimulai, dan setiap 10 step untuk titik-titik individu di dalam fase SWA.

In [19]:
from torch.optim.swa_utils import SWALR, AveragedModel

model = create_llm(2, 64, 2, 256, VOCAB)
optimizer = AdamW(model.parameters(), lr=3e-3)
swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=1e-2)      # fase SWA sengaja memakai lr konstan yang tinggi

SWA_START = P["swa_start"]
phase_vals = []
for step in range(P["swa_steps"]):
    model.train()
    x = overfit_batch(step)
    loss = model(input_ids=x, labels=x).loss
    check_finite(loss, "SWA")
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    if step == SWA_START:
        val_at_swa_start = eval_loss(model)
    if step > SWA_START:
        swa_model.update_parameters(model)
        swa_scheduler.step()
        if (step - SWA_START) % 10 == 0:
            phase_vals.append(eval_loss(model))         # titik individu dari fase SWA yang sama

print(f"saat fase SWA dimulai (step {SWA_START})        : val loss {val_at_swa_start:.3f}")
print(f"titik individu fase SWA ({len(phase_vals)} cek)       : val loss rata-rata {np.mean(phase_vals):.3f} "
      f"[{min(phase_vals):.2f}-{max(phase_vals):.2f}]")
print(f"titik terakhir (step {P['swa_steps'] - 1})                : val loss {eval_loss(model):.3f}")
print(f"rata-rata bobot SWA ({int(swa_model.n_averaged)} titik)        : val loss {eval_loss(swa_model.module):.3f}")
del model, swa_model, optimizer, swa_scheduler
free_gpu()

saat fase SWA dimulai (step 200)        : val loss 6.990
titik individu fase SWA (9 cek)       : val loss rata-rata 7.807 [7.22-8.57]
titik terakhir (step 299)                : val loss 8.774
rata-rata bobot SWA (99 titik)        : val loss 7.635


Klaim inti SWA adalah bahwa rata-rata *bobot* dari satu fase lebih baik daripada titik-titik individu dari fase yang sama, jadi bandingkan baris terakhir dengan baris kedua. Dengan lr tinggi, titik individu memantul sehingga val loss-nya berubah-ubah (lihat rentangnya); rata-rata SWA mewakili pusat pantulan itu. Pada validasi lokal kami (CPU dan MPS), rata-rata SWA lebih baik daripada rata-rata titik individu, tetapi tidak lebih baik daripada titik individu terbaik. Pada profil GPU, ia juga jauh lebih baik daripada titik terakhir.

Baris pertama adalah pembanding yang lebih ketat, dan hasilnya bergantung pada lintasan. Pada profil GPU kami, model saat fase SWA dimulai justru punya val loss terendah; pada profil `cpu-kecil`, titik itu lebih buruk daripada semua baris lain. Di rezim hafalan, setiap step tambahan menambah hafalan dan lr tinggi menambah kerusakan, sehingga rata-rata SWA bisa jadi hanya menebus sebagian. Jadi eksperimen mini ini memperlihatkan mekanika SWA, bukan bukti bahwa SWA mengalahkan berhenti lebih awal. Klaim manfaat SWA di literatur berasal dari training standar pada data yang jauh lebih besar, bukan dari rezim hafalan ekstrem seperti ini.

Dua catatan teknis. Rata-rata bobot hanya bermakna untuk titik-titik dari satu lintasan; dua model dari seed berbeda berada di lembah yang berbeda, dan rata-rata keduanya bisa menjadi model yang rusak. GPT-2 memakai LayerNorm, bukan BatchNorm, jadi langkah `update_bn` yang sering muncul di contoh SWA tidak diperlukan di sini.

### Sharpness-aware minimization (SAM)

Minimum yang "tajam" adalah lembah sempit: geser bobot sedikit dan loss melonjak. Minimum yang "datar" toleran terhadap geseran, dan toleransi itu sering dikaitkan dengan generalisasi yang lebih baik. SAM mencari bobot yang *lingkungannya* juga ber-loss rendah. Setiap step terdiri dari dua forward-backward: (1) hitung gradien di w, lalu naik sejauh ρ ke arah yang paling menaikkan loss, ke titik w + e(w); (2) hitung gradien di titik terburuk itu, kembali ke w, dan lakukan langkah optimizer biasa memakai gradien kedua.

> **Listing buku: kelas SAM**
>
> ```python
> def __init__(self, params, base_optimizer, rho=0.05):
>     self.rho = rho
>     self.base_optimizer = base_optimizer(params)
> def step(self):
>     grad_norm = self._grad_norm()
>     ...  # bobot dinaikkan dengan p.add_(e_w)
>     self.base_optimizer.step()
> ```
>
> `_grad_norm` tidak pernah didefinisikan (`AttributeError`, teramati), dan karena `super().__init__()` tidak dipanggil, `self.param_groups` pun tidak ada. Secara algoritma lebih fatal: forward-backward kedua tidak pernah dilakukan, dan bobot tidak pernah dikembalikan dari w + e(w), sehingga model melangkah dari titik terburuknya memakai gradien lama.

Kelas di bawah memperbaiki ketiganya dengan dua metode eksplisit, `first_step` dan `second_step`, lalu menguji satu hal yang bisa dicek secara pasti: dengan lr = 0, satu step SAM penuh harus mengembalikan bobot tepat ke titik asal.

In [20]:
class SAM(torch.optim.Optimizer):
    """SAM dua tahap (Foret dkk., 2021) di atas optimizer apa pun."""

    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        super().__init__(params, dict(rho=rho, **kwargs))
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups       # lr dan weight decay dipakai bersama
        self.defaults.update(self.base_optimizer.defaults)

    def _grad_norm(self):
        norms = [p.grad.norm(2) for group in self.param_groups for p in group["params"] if p.grad is not None]
        return torch.norm(torch.stack(norms), 2)

    @torch.no_grad()
    def first_step(self):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None:
                    continue
                self.state[p]["w"] = p.detach().clone()          # simpan titik asal
                p.add_(p.grad * scale)                          # naik ke w + e(w)

    @torch.no_grad()
    def second_step(self):
        for group in self.param_groups:
            for p in group["params"]:
                if "w" in self.state[p]:
                    p.copy_(self.state[p].pop("w"))             # kembali tepat ke w
        self.base_optimizer.step()                              # melangkah dengan gradien dari w + e(w)


# Cek mekanika: dengan lr = 0, satu step SAM penuh harus mengembalikan bobot tepat ke titik asal.
check_model = create_llm(2, 64, 2, 256, VOCAB, dropout=0.0, seed=0)
before = [p.detach().clone() for p in check_model.parameters()]
check_opt = SAM(check_model.parameters(), AdamW, rho=0.5, lr=0.0)
x = overfit_batch(0)
check_model(input_ids=x, labels=x).loss.backward()
check_opt.first_step()
moved = max((p - b).abs().max().item() for p, b in zip(check_model.parameters(), before))
check_opt.zero_grad(set_to_none=True)
check_model(input_ids=x, labels=x).loss.backward()
check_opt.second_step()
restored = all(torch.equal(p, b) for p, b in zip(check_model.parameters(), before))
print(f"pergeseran maksimum saat di w + e(w): {moved:.2e} | kembali tepat ke w: {restored}")
assert moved > 0 and restored
del check_model, check_opt, before
free_gpu()

pergeseran maksimum saat di w + e(w): 8.27e-02 | kembali tepat ke w: True


Bobot berpindah saat berada di w + e(w), lalu kembali bit demi bit ke w. Sekarang SAM dilatih dengan protokol rezim hafalan dan dibandingkan dengan dua model AdamW:

- **AdamW tanpa berhenti**: model tanpa regularisasi dari tabel tadi (seed 42, jumlah step sama);
- **AdamW yang berhenti di train loss setara SAM**: kontrol penting, karena model yang hafalannya lebih sedikit cenderung terlihat lebih datar dengan sendirinya, apa pun penyebabnya.

"Tajam" diukur secara langsung: geser seluruh bobot sejauh r (norma L2) ke delapan arah acak yang sama untuk ketiga model, lalu catat rata-rata kenaikan train loss. ρ = 0,5 dipakai untuk model mini ini (buku: 0,05), dan nilainya tidak di-tune. Perhatikan juga harganya: SAM memakai dua forward-backward per step, jadi dengan jumlah step yang sama compute-nya dua kali lipat.

In [21]:
# === UNDER THE HOOD: seberapa datar minimum yang ditemukan SAM? ===
SAM_RHO = 0.5                      # buku: 0.05; dipilih untuk model mini ini, belum di-tune


def train_sam(steps, rho, lr=1e-3, seed=42):
    model = create_llm(2, 64, 2, 256, VOCAB, dropout=0.0, seed=seed)
    init_norm = weight_norm(model)
    optimizer = SAM(model.parameters(), AdamW, rho=rho, lr=lr)
    for step in range(steps):
        model.train()
        x = overfit_batch(step)
        loss = model(input_ids=x, labels=x).loss          # forward-backward 1: gradien di w
        check_finite(loss, "SAM")
        loss.backward()
        optimizer.first_step()                            # naik ke w + e(w)
        optimizer.zero_grad(set_to_none=True)
        model(input_ids=x, labels=x).loss.backward()      # forward-backward 2: gradien di w + e(w)
        optimizer.second_step()                           # kembali ke w, lalu langkah AdamW
        optimizer.zero_grad(set_to_none=True)
    return summarize(model, init_norm, steps)


@torch.no_grad()
def sharpness(model, radius, n_dirs=8):
    """Rata-rata kenaikan train loss saat bobot digeser sejauh `radius` (norma L2) ke arah acak."""
    base_loss = eval_loss(model, OVERFIT_BATCHES)
    params = list(model.parameters())
    saved = [p.detach().clone() for p in params]
    rises = []
    for k in range(n_dirs):
        gen = torch.Generator().manual_seed(1000 + k)                  # arah yang sama untuk semua model
        dirs = [torch.randn(p.shape, generator=gen).to(p.device) for p in params]
        scale = radius / torch.sqrt(sum((d ** 2).sum() for d in dirs))
        for p, d in zip(params, dirs):
            p.add_(d * scale)
        rises.append(eval_loss(model, OVERFIT_BATCHES) - base_loss)
        for p, s in zip(params, saved):
            p.copy_(s)
    return float(np.mean(rises))


r_sam = train_sam(P["overfit_steps"], rho=SAM_RHO)
r_early = train_overfit(P["overfit_steps"], stop_below=r_sam["train"])
rows = (("AdamW, tanpa berhenti", BASELINE), ("AdamW, berhenti di train ~ SAM", r_early), (f"SAM, rho={SAM_RHO}", r_sam))
print(f"{'model':<31} {'step':>4} {'train':>6} {'val':>6} {'||w||':>6} | kenaikan train loss saat digeser sejauh r")
for name, r in rows:
    rises = "  ".join(f"r={a}: {sharpness(r['model'], a):+.3f}" for a in (2.5, 5.0, 10.0))
    print(f"{name:<31} {r['steps']:>4} {r['train']:6.3f} {r['val']:6.3f} {r['wnorm']:6.1f} | {rises}")
del r_sam["model"], r_early["model"]
free_gpu()

model                           step  train    val  ||w|| | kenaikan train loss saat digeser sejauh r
AdamW, tanpa berhenti            500  0.412  8.239   66.0 | r=2.5: +0.073  r=5.0: +0.323  r=10.0: +1.355
AdamW, berhenti di train ~ SAM   190  3.177  6.176   45.0 | r=2.5: +0.041  r=5.0: +0.153  r=10.0: +0.544
SAM, rho=0.5                     500  3.177  6.173   48.9 | r=2.5: +0.008  r=5.0: +0.030  r=10.0: +0.119


Kolom kanan adalah definisi operasional "tajam" dan "datar". Pada validasi lokal kami muncul dua hal yang perlu dibedakan. Pertama, SAM jauh lebih datar daripada AdamW tanpa berhenti, dan val loss-nya jauh lebih baik. Kedua, kontrol di baris tengah, yaitu AdamW yang berhenti ketika train loss-nya setara SAM, mencapai val loss di kisaran yang sama dengan SAM (di run kami selisihnya hanya 0,05–0,2 nats, kadang lebih baik dan kadang lebih buruk), tetapi tetap beberapa kali lebih tajam.

Artinya, kedataran yang ditemukan SAM memang efek SAM sendiri, bukan sekadar akibat hafalan yang lebih sedikit. Sebaliknya, perbaikan val loss dalam eksperimen ini bisa dicapai juga dengan berhenti lebih awal; kedataran tidak otomatis berarti val loss lebih baik. Ukuran ini juga bergantung pada skala: model dengan ‖w‖ lebih besar "merasakan" geseran r yang sama sebagai geseran yang relatif lebih kecil. Karena itu kolom ‖w‖ ikut dicetak; di run kami norma SAM hanya sekitar 5–15% lebih besar daripada kontrolnya, terlalu kecil untuk menjelaskan selisih kedataran beberapa kali lipat.

### Differential privacy (DP)

Differential privacy menjanjikan bahwa ada atau tidaknya satu contoh di data training hampir tidak bisa dibedakan dari model hasilnya. Resep DP-SGD: (1) hitung gradien **per contoh**, (2) potong norma setiap gradien ke batas C sehingga pengaruh satu contoh terbatas, (3) jumlahkan, tambahkan derau Gaussian berskala σ · C, lalu rata-ratakan.

> **Listing buku: `DPOptimizer`**
>
> ```python
> def step(self):
>     torch.nn.utils.clip_grad_norm_(self.param_groups[0]['params'], self.max_grad_norm)
>     for p in self.param_groups[0]['params']:
>         noise = torch.randn_like(p.grad) * self.noise_multiplier
>         p.grad.add_(noise)
> ```
>
> Tanpa `super().__init__()`, `self.param_groups` tidak ada (`AttributeError`, teramati). Seandainya itu diperbaiki, `step()` tetap tidak pernah memperbarui parameter, dan clipping dilakukan pada gradien *batch*, bukan per contoh — satu contoh ekstrem masih bisa mendominasi. Itu bukan DP.

Versi di bawah menghitung gradien per contoh dengan cara paling sederhana, yaitu satu backward per contoh, lalu menyerahkan gradien yang sudah diprivatisasi ke AdamW biasa. Kita bandingkan σ = 0 (hanya clipping per contoh) dengan σ = 1, pada batch 48 contoh.

In [22]:
class DPOptimizer:
    """Mekanika DP-SGD: gradien per contoh dipotong ke norma C, dijumlah, diberi derau N(0, (sigma*C)^2),
    dirata-rata, lalu dipakai optimizer biasa. BUKAN jaminan privasi terhitung (tanpa accountant)."""

    def __init__(self, model, base_optimizer, noise_multiplier=1.0, max_grad_norm=1.0):
        self.model = model
        self.base_optimizer = base_optimizer
        self.noise_multiplier = noise_multiplier
        self.max_grad_norm = max_grad_norm
        self.params = [p for p in model.parameters() if p.requires_grad]

    def step(self, batch):
        summed = [torch.zeros_like(p) for p in self.params]
        n_clipped = 0
        for x in batch:                                      # satu contoh per backward = gradien per contoh
            self.model.zero_grad(set_to_none=True)
            loss = self.model(input_ids=x[None], labels=x[None]).loss
            check_finite(loss, "DP-SGD")
            loss.backward()
            norm = torch.sqrt(sum((p.grad ** 2).sum() for p in self.params))
            n_clipped += int(norm.item() > self.max_grad_norm)
            factor = torch.clamp(self.max_grad_norm / (norm + 1e-6), max=1.0)
            for s, p in zip(summed, self.params):
                s.add_(p.grad * factor)
        for s, p in zip(summed, self.params):
            noise = torch.randn_like(s) * self.noise_multiplier * self.max_grad_norm
            p.grad = (s + noise) / len(batch)
        self.base_optimizer.step()
        self.base_optimizer.zero_grad(set_to_none=True)
        return n_clipped


def run_dp(noise_multiplier, steps):
    model = create_llm(2, 64, 2, 256, VOCAB, dropout=0.0, seed=42)
    dp = DPOptimizer(model, AdamW(model.parameters(), lr=1e-3), noise_multiplier=noise_multiplier)
    start = eval_loss(model, OVERFIT_BATCHES)
    clipped = 0
    for step in range(steps):
        model.train()
        clipped += dp.step(OVERFIT)                          # 48 contoh per step
    return start, eval_loss(model, OVERFIT_BATCHES), eval_loss(model), clipped / (steps * len(OVERFIT))


print(f"norma derau per step ~ sigma*C*sqrt(d)/B = 1*1*sqrt({N_PARAMS:,})/48 = {math.sqrt(N_PARAMS) / 48:.1f}; "
      f"norma sinyal (rata-rata gradien terpotong) <= C = 1")
for sigma in (0.0, 1.0):
    start, train, val, frac = run_dp(sigma, P["dp_steps"])
    print(f"noise_multiplier {sigma}: train {start:.3f} -> {train:.3f} | val {val:.3f} | "
          f"{frac:.0%} gradien per contoh terpotong | {P['dp_steps']} step")
free_gpu()

norma derau per step ~ sigma*C*sqrt(d)/B = 1*1*sqrt(366,336)/48 = 12.6; norma sinyal (rata-rata gradien terpotong) <= C = 1
noise_multiplier 0.0: train 8.283 -> 5.333 | val 6.390 | 100% gradien per contoh terpotong | 40 step
noise_multiplier 1.0: train 8.283 -> 7.376 | val 7.438 | 100% gradien per contoh terpotong | 40 step


Baris pertama menghitung arti σ = 1 sebelum training dimulai: dengan 0,37 juta parameter dan batch 48, norma derau per step sekitar 12,6, sedangkan norma sinyal paling besar 1. Derau kira-kira 12 kali lebih besar daripada sinyal terkuat yang mungkin. Kolom terakhir menunjukkan berapa banyak gradien per contoh yang melewati batas C; jika hampir semuanya terpotong, C = 1 aktif membatasi pengaruh setiap contoh. Hasil training mengikuti hitungan derau tadi: dengan σ = 0 train loss turun normal, sedangkan dengan σ = 1 loss masih turun, tetapi jauh lebih lambat. Inilah harga DP: sinyal baru jelas mengalahkan derau kalau batch-nya sangat besar (ribuan contoh), karena jumlah gradien tumbuh sebanding dengan B sedangkan derau yang ditambahkan tetap.

Batasnya perlu jelas: sel ini hanya mekanika. Tanpa sampling acak (Poisson) dan tanpa *privacy accountant* yang menghitung ε, ia tidak memberi jaminan privasi apa pun. Untuk klaim privasi, gunakan pustaka seperti Opacus.

### Fast gradient sign method (FGSM)

FGSM bukan regularizer langsung. Ia membuat contoh adversarial: input yang digeser sedikit ke arah tanda gradien, yaitu arah yang paling cepat menaikkan loss. Contoh seperti itu kemudian bisa dipakai untuk *adversarial training*.

> **Listing buku: FGSM untuk citra**
>
> ```python
> perturbed_image = image + epsilon * sign_data_grad
> perturbed_image = torch.clamp(perturbed_image, 0, 1)
> ```
>
> Fungsi ini benar untuk piksel bernilai 0–1, tetapi tidak cocok untuk teks: token id tidak punya gradien, dan `clamp(0, 1)` akan merusak embedding yang sebagian nilainya negatif. Ini keterbatasan konsep, bukan error yang dijalankan. Versi teks di bawah menyerang ruang embedding tanpa clamp.

Korbannya adalah model tanpa regularisasi dari tabel tadi, diuji di seluruh 512 blok validasi. Pembandingnya derau acak dengan besaran yang persis sama: setiap koordinat embedding digeser ±ε, dengan ε = 5% simpangan baku embedding. Satu-satunya perbedaan adalah arah.

In [23]:
def fgsm_embedding_attack(embd, epsilon, data_grad):
    return embd + epsilon * data_grad.sign()          # tanpa clamp: embedding tidak punya rentang [0, 1]


victim = BASELINE["model"]
victim.eval()                                         # dropout mati; gradien hanya untuk input
gen = torch.Generator().manual_seed(0)
clean, adversarial, random_dir = [], [], []
for x in VAL_BATCHES:
    embd = victim.transformer.wte(x).detach().requires_grad_(True)
    loss = victim(inputs_embeds=embd, labels=x).loss
    (grad,) = torch.autograd.grad(loss, embd)         # gradien terhadap embedding, bukan terhadap bobot
    eps = 0.05 * embd.std().item()
    random_sign = torch.randn(embd.shape, generator=gen).sign().to(DEVICE)
    with torch.no_grad():
        clean.append(loss.item())
        adversarial.append(victim(inputs_embeds=fgsm_embedding_attack(embd, eps, grad), labels=x).loss.item())
        random_dir.append(victim(inputs_embeds=embd + eps * random_sign, labels=x).loss.item())

print(f"val loss bersih          : {np.mean(clean):.3f}")
print(f"+ derau acak (eps sama)  : {np.mean(random_dir):.3f}  ({np.mean(random_dir) - np.mean(clean):+.3f})")
print(f"+ FGSM (eps sama)        : {np.mean(adversarial):.3f}  ({np.mean(adversarial) - np.mean(clean):+.3f})")

val loss bersih          : 8.239
+ derau acak (eps sama)  : 8.233  (-0.006)
+ FGSM (eps sama)        : 9.771  (+1.533)


Arah acak hampir tidak mengubah loss, sedangkan arah tanda gradien menaikkannya dengan jelas, padahal besaran geserannya identik, baik dalam norma L∞ maupun L2. Pesannya: model punya arah-arah rapuh yang hampir tidak pernah ditemukan oleh derau biasa. Adversarial training memanfaatkannya dengan ikut melatih model pada input hasil `fgsm_embedding_attack`; langkah itu tidak dijalankan di notebook ini.

### Lookahead optimizer

Lookahead menyimpan dua salinan bobot. Bobot *fast* diperbarui optimizer dasar setiap step. Setiap k step, bobot *slow* bergerak sebagian (α) ke arah fast, lalu fast di-reset ke slow. Efeknya, lintasan bobot menjadi lebih halus. Kelas buku berfungsi apa adanya, jadi kita jalankan tanpa perubahan dan memeriksa siklusnya dengan `assert`:

In [24]:
class Lookahead(torch.optim.Optimizer):              # listing buku, tidak diubah
    def __init__(self, optimizer, k=5, alpha=0.5):
        self.optimizer = optimizer
        self.k = k
        self.alpha = alpha
        self.step_counter = 0
        self.slow_weights = [
            [p.clone().detach() for p in group['params']]
            for group in optimizer.param_groups
        ]

    def step(self):
        self.step_counter += 1
        self.optimizer.step()
        if self.step_counter % self.k == 0:
            for group, slow_weights in zip(self.optimizer.param_groups, self.slow_weights):
                for p, q in zip(group['params'], slow_weights):
                    p.data.mul_(self.alpha).add_(q, alpha=1.0 - self.alpha)
                    q.data.copy_(p.data)


model = create_llm(2, 64, 2, 256, VOCAB)
inner = AdamW(model.parameters(), lr=3e-3)
look = Lookahead(inner, k=5, alpha=0.5)


@torch.no_grad()
def fast_slow_gap():
    return max(float((p - q).abs().max())
               for group, slow in zip(inner.param_groups, look.slow_weights)
               for p, q in zip(group["params"], slow))


gaps = {}
for step in range(1, 21):
    model.train()
    x = overfit_batch(step)
    loss = model(input_ids=x, labels=x).loss
    loss.backward()
    look.step()
    inner.zero_grad()                                # zero_grad ke optimizer dalam, bukan ke wrapper
    gaps[step] = fast_slow_gap()
    if step in (4, 5, 19, 20):
        print(f"step {step:>2}: loss {loss.item():.3f} | maks |fast - slow| = {gaps[step]:.4f}"
              + ("  <- sinkronisasi" if step % 5 == 0 else ""))

assert all(gaps[s] == 0.0 for s in (5, 10, 15, 20)) and all(g > 0 for s, g in gaps.items() if s % 5)
del model, inner, look
free_gpu()

step  4: loss 7.645 | maks |fast - slow| = 0.0120
step  5: loss 7.435 | maks |fast - slow| = 0.0000  <- sinkronisasi
step 19: loss 5.987 | maks |fast - slow| = 0.0132
step 20: loss 5.782 | maks |fast - slow| = 0.0000  <- sinkronisasi


Selisih fast–slow tumbuh selama empat step, lalu kembali tepat ke nol di setiap step kelipatan 5: fast ditarik ke titik tengah antara dirinya dan slow (α = 0,5), dan slow menyalin titik temu itu. Satu jebakan: kelas ini tidak memanggil `super().__init__()`, sehingga `look.zero_grad()` gagal dengan `AttributeError` (teramati). Karena itu `zero_grad` dan scheduler harus diarahkan ke optimizer dalam (`inner`), bukan ke wrapper. Sel ini memeriksa mekanika; manfaat Lookahead dibanding AdamW biasa tidak diuji di sini.

## Ringkasan

- **Rezim hafalan dulu, regularisasi kemudian.** Dengan 48 blok yang diputar ratusan kali, model tanpa regularisasi menghafal: train loss jauh di bawah patokan unigram, val loss di atasnya. Di rezim ini efek setiap teknik bisa dibaca dari train loss, val loss, gap, dan ‖w‖, dengan bobot awal, data, urutan batch, dan jumlah step yang sama, serta rentang antar-seed sebagai ukuran variasi.
- **L2 ≠ weight decay di Adam.** Kesetaraannya hanya berlaku untuk SGD (dan untuk `Adam(weight_decay=...)`, yang memang L2). AdamW memisahkan peluruhan dari normalisasi Adam, sehingga λ yang sama memberi efek yang sangat berbeda.
- **Dropout di PyTorch adalah inverted dropout.** Penskalaan terjadi saat training; saat eval dropout adalah identitas, asalkan `model.eval()` dipanggil.
- **Bug sunyi lebih mahal daripada error.** Encoder tanpa causal mask memberi loss yang "terlalu bagus"; baris `+ i * increase` mematikan jadwal dropout per layer tanpa traceback; derau pada token id mengganti sekitar separuh token. Ketiganya hanya tertangkap dengan mengukur.
- **Teknik yang lebih baru perlu pembanding yang jujur.** Pada validasi lokal kami, rata-rata bobot SWA lebih baik daripada rata-rata titik individu di fasenya, tetapi tidak mengalahkan titik individu terbaik. SAM menemukan minimum yang jauh lebih datar bahkan dibanding AdamW dengan train loss setara, sementara keuntungan val loss-nya bisa disamai dengan berhenti lebih awal. DP dengan σ = 1 pada batch 48 memperlambat belajar karena derau sekitar 12 kali sinyal terkuat. FGSM merusak loss jauh lebih parah daripada derau acak berbesaran sama. Lookahead hanya diperiksa mekanikanya.

Kesimpulan di atas berlaku untuk model mini, korpus kecil, dan jumlah seed yang diuji; angka persisnya bisa berbeda di GPU Anda.

**Berikutnya:** Bab 10 membahas checkpointing dan recovery — cara agar training panjang yang gagal di tengah jalan tidak harus diulang dari nol.